# Update agents

In [ ]:
# Ensure the database path is in environment variables
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["DATABASE_URL"]

In [ ]:
# Add the backend path to sys.path
import sys

backend_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", ".."))
if backend_path not in sys.path:
    sys.path.append(backend_path)

In [ ]:
# Initialize database orm model
from db.init_db import init_db

init_db()

In [ ]:
# Test querying the database
from db.session import LocalSession
from models.project import Project

with LocalSession() as db:
    projects = db.query(Project).all()
    print(projects)

In [ ]:
# Confirm there are no agents yet
from models.project import Agent

with LocalSession() as db:
    agents = db.query(Agent).all()
    print(agents)

In [ ]:
# Create an agent object and add it to the database
from schemas.agent import AgentCreate

agent_create = AgentCreate(
    id="chiefofstaff",
    name="chiefofstaff",
    role="assistant",
    model="o3-mini",
    params={},
)

with LocalSession() as db:
    agent = Agent(**agent_create.model_dump())
    db.add(agent)
    db.commit()
    db.refresh(agent)

with LocalSession() as db:
    agents = db.query(Agent).all()
    print(agents)

In [ ]:
# Update projects to include the new agent
with LocalSession() as db:
    agent = db.get(Agent, "chiefofstaff")
    if not agent:
        raise ValueError("Agent not found")
    
    projects = db.query(Project).all()
    for project in projects:
        project.current_agent_id = agent.id
        db.commit()
        db.refresh(project)

In [ ]:
# Check to see if the agent is now in the projects
with LocalSession() as db:
    projects = db.query(Project).all()
    for project in projects:
        print(f"{project=} {project.current_agent=}")